# Decay rate

A first-order decay, measured every two hours: fit the log of the concentration against time for the rate constant and the half-life.

In [ ]:
import csv
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

In [ ]:
with Path("data/decay.csv").open(encoding="utf-8") as table:
    rows = list(csv.DictReader(table))
hours = np.array([float(row["hour"]) for row in rows])
concentration = np.array([float(row["concentration"]) for row in rows])
len(rows)

In [ ]:
slope, intercept = np.polyfit(hours, np.log(concentration), 1)
rate = -slope
half_life = np.log(2) / rate
print(f"rate: {rate:.4f} per hour")
print(f"half-life: {half_life:.2f} hours")

On a log scale a first-order decay is a straight line, so the right panel shows how well the model holds.

In [ ]:
smooth = np.linspace(hours[0], hours[-1], 200)
fitted = np.exp(intercept + slope * smooth)

figure, (linear, logarithmic) = plt.subplots(1, 2, figsize=(10, 4), dpi=100)
for axes in (linear, logarithmic):
    axes.scatter(hours, concentration, label="measured")
    axes.plot(smooth, fitted, color="tab:orange", label="fit")
    axes.set_xlabel("hour")
    axes.set_ylabel("concentration")
logarithmic.set_yscale("log")
linear.legend(title=f"half-life {half_life:.2f} h")
figure.tight_layout()
figure.savefig("figure.png", metadata={"Software": None})
plt.show()

The figure is a picture of numbers, so keep the numbers: the table behind it, and what each panel actually drew, read back from the figure.

In [ ]:
with Path("curve.csv").open("w", encoding="utf-8", newline="") as table:
    writer = csv.writer(table)
    writer.writerow(["hour", "measured", "fitted"])
    for hour, measured, value in zip(hours, concentration, np.exp(intercept + slope * hours), strict=True):
        writer.writerow([f"{hour:g}", f"{measured:.2f}", f"{value:.4f}"])


def drawn(axes):
    series = [(collection.get_label(), collection.get_offsets()) for collection in axes.collections]
    series += [(line.get_label(), line.get_xydata()) for line in axes.lines]
    return {
        "xlabel": axes.get_xlabel(),
        "ylabel": axes.get_ylabel(),
        "yscale": axes.get_yscale(),
        "series": {label: np.round(np.asarray(points), 4).tolist() for label, points in series},
    }


Path("figure.json").write_text(json.dumps({"panels": [drawn(axes) for axes in figure.axes]}) + "\n", encoding="utf-8")

In [ ]:
Path("result.txt").write_text(
    f"rate: {rate:.4f} per hour\nhalf-life: {half_life:.2f} hours\nC0: {np.exp(intercept):.2f}\n",
    encoding="utf-8",
)